# Lesson 04: MCP Security — OWASP Top 10, Tool Pinning & CIMD vs. DCR

In this notebook, you will:
1. Validate **Client ID Metadata Documents (CIMD)** vs. **Dynamic Client Registration (DCR)**.
2. Verify JWT `iss`, `aud`, and `scope` (`tool:greet`) claims with `PyJWT` using local ephemeral RSA/HMAC keys.
3. Detect Tool Poisoning rug pulls via SHA-256 schema pinning and apply Spotlighting datamarking.

In [ ]:
!pip install -q -U PyJWT cryptography

In [ ]:
import jwt
import time

SECRET = "demo-ephemeral-signing-key-32-bytes"
MCP_SERVER_URL = "http://127.0.0.1:3001/mcp"
ISSUER = "https://auth.example.com/"

# Mint a valid MCP-scoped JWT
valid_token = jwt.encode(
    {
        "iss": ISSUER,
        "aud": MCP_SERVER_URL,
        "sub": "user-42",
        "client_id": "https://app.example.com/client-metadata.json",
        "scope": "tool:greet",
        "exp": int(time.time()) + 300,
    },
    SECRET,
    algorithm="HS256",
)

decoded = jwt.decode(valid_token, SECRET, algorithms=["HS256"], audience=MCP_SERVER_URL, issuer=ISSUER)
print("Verified JWT claims:", decoded)

# Reject token issued for a different audience (token passthrough protection)
try:
    jwt.decode(valid_token, SECRET, algorithms=["HS256"], audience="https://wrong-server.example.com/mcp", issuer=ISSUER)
except jwt.InvalidAudienceError as exc:
    print("Blocked token passthrough:", exc)

### Challenge Exercise
Write a function `check_cimd_document(doc: dict, client_id_url: str) -> bool` that verifies `doc['client_id'] == client_id_url`, ensures all `redirect_uris` are loopback or HTTPS, and confirms `token_endpoint_auth_method == 'none'` for public native clients.